# LIME with learned generative masking

**MedLIME Learning Track — Notebook 07/12** · Curriculum weeks 21–24 · Suggested time: 60–100 minutes

**Before starting:** Notebooks 04 and 06.

**Learning goal:** Replace black masking with learned image reconstruction inside LIME.

> **Important:** All images below are synthetic visual teaching phantoms, **not real medical images**. The educational implementations are **not** the original MedLIME implementation, are not evaluated for clinical use, and do not establish diagnostic or explanation reliability.


In [ ]:
from pathlib import Path
import sys
here = Path.cwd()
for candidate in (here, here.parent, here.parent.parent):
    if (candidate / 'medlime_tutorials.py').exists():
        sys.path.insert(0, str(candidate.resolve())); break
else:
    raise FileNotFoundError('Open Jupyter from the MedLIME folder or its notebooks/ subfolder')
import numpy as np
import matplotlib.pyplot as plt
from medlime_tutorials import *
np.set_printoptions(precision=3, suppress=True)


## Full perturbation pipeline
For each sampled patch mask \(z\), reconstruct a partially hidden query, pass it to the black-box model, then regress its scores on \(z\). This is a **toy analogue** of the paper's Generative Masking (GM), not a replacement for the paper's pretrained MAE.

In [ ]:
model,_=train_inpainter(n=96,epochs=8,seed=10)
x,lesion=synthetic_image(seed=13)
black=lime_explain(x,n=140,seed=8,fill='zero')
learned=lime_explain(x,n=140,seed=8,fill='learned',inpainter=model)
show_images([x,lesion.astype(float),black['map'],learned['map']],
 ['original','ground truth','LIME: black mask','LIME: learned mask'],cmap='viridis')
plt.show()

## Quantify both localization and surrogate agreement
AUPRC uses lesion pixels as positives; R² quantifies how well the fitted surrogate predicts the black-box scores of generated perturbations. Neither alone establishes causal faithfulness.

In [ ]:
for name,r in [('black',black),('learned',learned)]:
 print(name,'AUPRC=',round(auprc(lesion,r['map']),3),'R²=',round(r['r2'],3),
       'score range=',tuple(np.round([r['scores'].min(),r['scores'].max()],3)))
print('Top patch (black):',int(np.argmax(black['coef'])))
print('Top patch (learned):',int(np.argmax(learned['coef'])))

## Pairwise ablation
Evaluate multiple query seeds, always with the same perturbation masks for each pair. It is possible for the tiny inpainter to **lose** to black masking; the important scientific skill is to report it without hiding failures.

In [ ]:
rows=[]
for seed in [2,3,4,5]:
    q,gt=synthetic_image(seed,True)
    a=lime_explain(q,n=96,seed=22,fill='zero')
    b=lime_explain(q,n=96,seed=22,fill='learned',inpainter=model)
    rows.append([seed,auprc(gt,a['map']),auprc(gt,b['map'])])
print('seed | zero-fill AUPRC | learned-fill AUPRC')
for r in rows: print(r[0],round(r[1],3),round(r[2],3))

## What you should now understand
- Generative masking changes the perturbation *distribution* before regression.
- Identical masks support a controlled comparison of filling strategies.
- A teaching reconstructor may not improve localization; report both successes and failures.

## Try it yourself (do not just read the code)
1. Compare the learned reconstructor with `fill='blur'` on the same sample.
2. Choose one sample with worse learned-mask AUPRC and explain a likely cause.

## Exit ticket
**Question:** What does the LIME regression assume about reconstruction consistency as the binary mask changes?

Record your answers in a new Markdown cell before moving to the next notebook. Many questions have more than one reasonable answer; explain your evidence.